# 04_main_model
KLEC 필러워드 기반 발화 품질 분류 (KLUE-RoBERTa)

**입력:** `clean_text` (필러워드 태그 제거된 순수 발화)

**출력:** Good(1) / Poor(0) 이진 분류

## 셀 1 — Google Drive 마운트 & 라이브러리 설치

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install transformers torch scikit-learn pandas numpy -q
print("설치 완료!")

## 셀 2 — 데이터 로드 & 확인

In [ ]:
import pandas as pd
import numpy as np

base_path = '/content/drive/MyDrive/presentation_data/'

train_df = pd.read_csv(base_path + 'train.csv')
val_df   = pd.read_csv(base_path + 'val.csv')
test_df  = pd.read_csv(base_path + 'test.csv')

print(f"Train: {len(train_df)}개")
print(f"Val:   {len(val_df)}개")
print(f"Test:  {len(test_df)}개")
print(f"\n컬럼: {list(train_df.columns)}")

# clean_text 샘플 확인 (태그가 없어야 정상)
print("\n[clean_text 샘플 3개]")
print(train_df['clean_text'].head(3).to_string())

# 레이블 분포
print("\n[Train 레이블 분포]")
print(train_df['label'].value_counts())
print(f"Good 비율: {train_df['label'].mean()*100:.1f}%")

## 셀 3 — 토크나이저 & 모델 로드

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = 'klue/roberta-small'
tokenizer = AutoTokenizer.from_pretrained(model_name)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"토크나이저 로드 완료!")
print(f"사용 디바이스: {device}")

## 셀 4 — Dataset & DataLoader

In [ ]:
from torch.utils.data import Dataset, DataLoader

class PresentationDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        # clean_text 우선 사용, 없으면 text 사용
        if 'clean_text' in df.columns:
            self.texts = df['clean_text'].fillna('').tolist()
        else:
            self.texts = df['text'].fillna('').tolist()
        self.labels    = df['label'].tolist()
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label':          torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_dataset = PresentationDataset(train_df, tokenizer)
val_dataset   = PresentationDataset(val_df,   tokenizer)
test_dataset  = PresentationDataset(test_df,  tokenizer)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=32)
test_loader  = DataLoader(test_dataset,  batch_size=32)

print(f"Train 배치 수: {len(train_loader)}")
print(f"Val   배치 수: {len(val_loader)}")
print("데이터셋 준비 완료!")

## 셀 5 — 모델 로드 & 학습 설정

In [ ]:
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

model = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=2
)
model = model.to(device)

epochs       = 5
optimizer    = AdamW(model.parameters(), lr=2e-5)
total_steps  = len(train_loader) * epochs
scheduler    = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=total_steps // 10,
    num_training_steps=total_steps
)

print("모델 로드 완료!")
print(f"총 학습 스텝: {total_steps}")

## 셀 6 — 학습 & 평가 함수

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

def train_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    total_loss, preds, labels = 0, [], []
    for batch in loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        label          = batch['label'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=label)
        loss    = outputs.loss
        loss.backward()
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        preds  += outputs.logits.argmax(dim=1).cpu().tolist()
        labels += label.cpu().tolist()

    return total_loss / len(loader), accuracy_score(labels, preds), f1_score(labels, preds)


def eval_epoch(model, loader, device):
    model.eval()
    total_loss, preds, labels = 0, [], []
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            label          = batch['label'].to(device)

            outputs     = model(input_ids=input_ids, attention_mask=attention_mask, labels=label)
            total_loss += outputs.loss.item()
            preds  += outputs.logits.argmax(dim=1).cpu().tolist()
            labels += label.cpu().tolist()

    return total_loss / len(loader), accuracy_score(labels, preds), f1_score(labels, preds)

print("함수 정의 완료!")

## 셀 7 — 학습 루프

In [ ]:
import os

save_path  = base_path + 'best_model'
best_val_f1 = 0
history     = []

for epoch in range(epochs):
    train_loss, train_acc, train_f1 = train_epoch(model, train_loader, optimizer, scheduler, device)
    val_loss,   val_acc,   val_f1   = eval_epoch(model, val_loader, device)

    history.append({
        'epoch': epoch + 1,
        'train_loss': train_loss, 'train_acc': train_acc, 'train_f1': train_f1,
        'val_loss':   val_loss,   'val_acc':   val_acc,   'val_f1':   val_f1
    })

    print(f"Epoch {epoch+1}/{epochs} "
          f"| Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} F1: {train_f1:.4f} "
          f"| Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} F1: {val_f1:.4f}")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        model.save_pretrained(save_path)
        tokenizer.save_pretrained(save_path)
        print(f"  → 최고 모델 저장! Val F1: {val_f1:.4f}")

print(f"\n학습 완료! 최고 Val F1: {best_val_f1:.4f}")

## 셀 8 — 학습 곡선 시각화

In [ ]:
import matplotlib.pyplot as plt

hist_df = pd.DataFrame(history)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(hist_df['epoch'], hist_df['train_loss'], label='Train Loss', marker='o')
axes[0].plot(hist_df['epoch'], hist_df['val_loss'],   label='Val Loss',   marker='o')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(hist_df['epoch'], hist_df['train_f1'], label='Train F1', marker='o')
axes[1].plot(hist_df['epoch'], hist_df['val_f1'],   label='Val F1',   marker='o')
axes[1].set_title('F1 Score')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.tight_layout()
plt.savefig(base_path + 'training_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print("그래프 저장 완료!")

## 셀 9 — Test 셋 최종 평가

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# 최고 모델 로드
best_model = AutoModelForSequenceClassification.from_pretrained(save_path)
best_model = best_model.to(device)

test_loss, test_acc, test_f1 = eval_epoch(best_model, test_loader, device)
print(f"=== Test 결과 ===")
print(f"Test Accuracy: {test_acc:.4f} ({test_acc*100:.1f}%)")
print(f"Test F1-score: {test_f1:.4f}")

# Classification report
best_model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in test_loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        outputs        = best_model(input_ids=input_ids, attention_mask=attention_mask)
        all_preds  += outputs.logits.argmax(dim=1).cpu().tolist()
        all_labels += batch['label'].tolist()

print("\n" + classification_report(all_labels, all_preds, target_names=['Poor(0)', 'Good(1)']))

# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Poor(0)', 'Good(1)'],
            yticklabels=['Poor(0)', 'Good(1)'])
plt.title('Confusion Matrix (Test)')
plt.ylabel('실제')
plt.xlabel('예측')
plt.tight_layout()
plt.savefig(base_path + 'confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print("평가 완료!")